# Full config

Every config field volcatenate exposes, organized by backend, with the paper default shown explicitly. Use this as a reference when you want to know what's available and what each field does. For a plain-English explanation of how each field actually propagates into the underlying model — including which values are silently pulled from the sample composition and what fallback chains apply — see the [config propagation reference](../config_options.md).

The minimal example is in [minimal_config.ipynb](minimal_config.ipynb).

## Sample composition

Most config fields don't matter unless you have a melt to feed them. Here's a synthetic basalt with every redox indicator populated, so we can demonstrate the strict `fo2_source` / `redox_source` settings later.

> This notebook builds the sample as a `MeltComposition` for compactness. In practice most users load samples from a CSV — see [sample_data.md](../sample_data.md) for the full input reference (CSV, Python dict, or `MeltComposition`).

In [ ]:
from volcatenate.composition import MeltComposition

morb = MeltComposition(
    sample="MORB",
    T_C=1200,
    # Major oxides (wt%)
    SiO2=50.5, TiO2=1.6, Al2O3=15.0, Cr2O3=0.04, FeOT=10.5,
    MnO=0.18, MgO=7.8, CaO=11.0, Na2O=2.8, K2O=0.16, P2O5=0.18,
    # Volatiles (wt%)
    H2O=0.5, CO2=0.04, S=0.12,
    # Redox indicators (use whichever you have; volcatenate will pick)
    Fe3FeT=0.10, dNNO=-0.5, dFMQ=0.25,
)
morb.to_dict()

## Top-level RunConfig

These control output destinations, logging, progress bars, and reproducibility bundles. Everything else lives on a nested per-backend sub-config.

In [ ]:
from volcatenate.config import RunConfig

config = RunConfig(
    output_dir=".",                      # where standardized CSVs land
    raw_output_dir="raw_tool_output",    # subdir for backend scratch files
    keep_raw_output=True,                # retain raw files after run
    verbose=False,                       # INFO logging to terminal
    log_file="",                         # optional debug-level log file
    show_progress=True,                  # rich progress bars
    save_bundle="",                      # path for reproducible JSON bundle
    bundle_comments="",                  # free-text run notes
)

## VESIcal

VESIcal is H₂O–CO₂ only; it does not model sulfur. The solubility model (Iacono-Marziano, Dixon, MagmaSat, etc.) is selected by **backend name** when calling `calculate_*`, not by a config field — see `VARIANT_MAP` in `volcatenate.backends.vesical`.

In [ ]:
from volcatenate.config import VESIcalConfig

config.vesical = VESIcalConfig(
    steps=101,                # pressure steps in the degassing path
    final_pressure=1.0,       # bar, lowest P in the path
    fractionate_vapor=0.0,    # 0 = closed-system, 1 = open-system
)

## VolFe

VolFe is the largest config surface — almost every field maps 1-to-1 to a VolFe internal model option. The propagation doc has the full mapping.

In [ ]:
from volcatenate.config import VolFeConfig

config.volfe = VolFeConfig(
    # ── Saturation ──────────────────────────────────────────
    sulfur_saturation=False,
    graphite_saturation=False,
    sulfur_is_sat="no",            # "yes" treats melt as S-saturated at start

    # ── Redox input ─────────────────────────────────────────
    fo2_column="Fe3FeT",            # which redox column to send to VolFe
    fo2_source="auto",              # "auto" | "fe3fet" | "dnno" | "dfmq"

    # ── Degassing geometry ──────────────────────────────────
    gassing_style="closed",         # "closed" or "open"
    gassing_direction="degas",      # "degas" or "regas"
    bulk_composition="melt-only",   # "melt-only" | "melt+vapor_wtg" | "melt+vapor_initialCO2"
    starting_p="Pvsat",             # "Pvsat" or "set"
    p_variation="polybaric",        # "polybaric" or "isobaric"
    t_variation="isothermal",       # "isothermal" or "polythermal"
    crystallisation="no",           # track crystallization during degassing

    # ── Iron / oxygen redox ─────────────────────────────────
    eq_fe="yes",                    # equilibrate Fe redox each step
    bulk_o="exc_S",                 # "exc_S" or "inc_S"
    calc_sat="fO2_melt",            # "fO2_melt" or "fO2_fX"

    # ── Species ─────────────────────────────────────────────
    coh_species="yes_H2_CO_CH4_melt",
    h2s_melt=True,
    species_x="Ar",                 # "Ar" or "Ne"
    h_speciation="none",

    # ── Oxygen fugacity ─────────────────────────────────────
    fo2_model="Kress91A",           # fO2-Fe3+/FeT relationship
    fmq_buffer="Frost91",
    nno_buffer="Frost91",

    # ── Bulk physical model ─────────────────────────────────
    density="DensityX",
    melt_composition="Basalt",

    # ── Solubility constants ────────────────────────────────
    co2_sol="MORB_Dixon95",
    h2o_sol="Basalt_Hughes24",
    h2_sol="Basalt_Hughes24",
    sulfide_sol="ONeill21dil",
    sulfate_sol="ONeill22dil",
    h2s_sol="Basalt_Hughes24",
    ch4_sol="Basalt_Ardia13",
    co_sol="Basalt_Hughes24",
    x_sol="Ar_Basalt_Hughes25",
    c_spec_comp="Basalt",
    h_spec_comp="MORB_HughesIP",

    # ── Saturation conditions ───────────────────────────────
    scss="ONeill21hyd",
    scas="Zajacz19_pss",

    # ── Fugacity coefficients ───────────────────────────────
    ideal_gas=False,
    y_co2="Shi92", y_so2="Shi92_Hughes23", y_h2s="Shi92_Hughes24",
    y_h2="Shaw64", y_o2="Shi92", y_s2="Shi92",
    y_co="Shi92", y_ch4="Shi92", y_h2o="Holland91", y_ocs="Shi92",
    y_x="ideal",

    # ── Equilibrium-constant models ─────────────────────────
    k_hog="Ohmoto97", k_hosg="Ohmoto97", k_osg="Ohmoto97",
    k_osg2="ONeill22", k_cog="Ohmoto97", k_cohg="Ohmoto97",
    k_ocsg="Moussallam19", k_cos="Holloway92",
    carbonylsulfide="COS",

    # ── Isotopes ────────────────────────────────────────────
    # All alpha_* / beta_factors only matter when isotopes="yes".
    isotopes="no",
    beta_factors="Richet77",
    alpha_h_ch4v_ch4m="no fractionation",
    alpha_h_h2v_h2m="no fractionation",
    alpha_h_h2ov_ohmm="Rust04",
    alpha_h_h2ov_h2om="Rust04",
    alpha_h_h2sv_h2sm="no fractionation",
    alpha_c_ch4v_ch4m="no fractionation",
    alpha_c_cov_com="no fractionation",
    alpha_c_co2v_co2t="Lee24",
    alpha_c_co2v_co2m="Blank93",
    alpha_c_co2v_co32mm="Lee24",
    alpha_s_h2sv_h2sm="no fractionation",
    alpha_so2_so4="Fiege15",
    alpha_h2s_s="Fiege15",

    # ── Numerical / runtime ─────────────────────────────────
    error=0.1,
    high_precision=False,
)

## EVo

EVo is run via three YAML files (`chem.yaml`, `env.yaml`, `output.yaml`) that volcatenate writes from this config plus the sample composition. Most users only touch `p_start`, `p_stop`, `gas_system`, and `fo2_source`; the rest are tuning settings.

In [ ]:
from volcatenate.config import EVoConfig

config.evo = EVoConfig(
    # ── Phase / system ──────────────────────────────────────
    gas_system="cohs",                # "cohs", "coh", "cos", etc.
    composition="basalt",             # "basalt" | "phonolite" | "rhyolite"
    fe_system=True,
    find_saturation=True,
    single_step=False,
    s_sat_warn=False,
    atomic_mass_set=False,
    ocs=False,
    run_type="closed",                # "closed" or "open" (open requires loss_frac < 1)

    # ── Pressure-stepping ───────────────────────────────────
    p_start=3000,                     # starting P (bar)
    p_stop=1,                         # final P (bar)
    dp_min=1, dp_max=100,
    mass=100,                         # system mass (g)
    wgt=0.00001,                      # initial gas weight fraction
    loss_frac=0.9999,                 # gas loss per step (open-system)

    # ── Atomic-mass volatile init (only when atomic_mass_set=True) ──
    atomic_h=500, atomic_c=200, atomic_s=4000, atomic_n=10,

    # ── Nitrogen / graphite ─────────────────────────────────
    nitrogen_set=False, nitrogen_start=0.0001,
    graphite_saturated=False, graphite_start=0.0001,

    # ── Redox initialization ────────────────────────────────
    # "auto" picks the best available source on the sample;
    # strict modes raise if the sample lacks the requested data.
    fo2_source="auto",                # "auto" | "fe3fet" | "buffer" | "absolute"
    fo2_buffer="FMQ",                 # "FMQ" | "NNO" | "IW"

    # Alternate fugacity entry points (ignored unless their *_set is True
    # or fo2_source explicitly enables them):
    fo2_set=False, fo2_start=0.0,     # absolute fO2 (bar)
    fh2_set=False, fh2_start=0.24,
    fh2o_set=False, fh2o_start=1000.0,
    fco2_set=False, fco2_start=1.0,

    # ── Solubility / model selections ───────────────────────
    h2o_model="burguisser2015",
    h2_model="gaillard2003",
    c_model="burguisser2015",
    co_model="armstrong2015",
    ch4_model="ardia2013",
    sulfide_capacity="oneill2020",
    sulfate_capacity="nash2019",
    scss="liu2007",
    n_model="libourel2003",
    density_model="spera2000",
    fo2_model="kc1991",
    fmq_model="frost1991",
)

## MAGEC

MAGEC runs as a MATLAB subprocess. The 14 numeric fields below correspond exactly to MAGEC's settings sheet. `redox_source` controls the wrapper-side fallback chain; in `"auto"` mode the wrapper may compute Fe³⁺/FeT from a buffer offset via Kress & Carmichael (1991) inversion if no Fe³⁺/FeT is available — see the propagation doc for the full cascade.

In [ ]:
from volcatenate.config import MAGECConfig

config.magec = MAGECConfig(
    # Paths — auto-detected at import; set if detection fails or you
    # have multiple installations.
    solver_dir="",                # path to MAGEC_Solver_v1b.p directory
    matlab_bin="",                # path to matlab binary

    # ── Saturation toggles (1=on, 0=off) ────────────────────
    sulfide_sat=0, sulfate_sat=0, graphite_sat=0,

    # ── Petrologic models (integers per MAGEC settings sheet) ──
    fe_redox=1,                   # 1=Sun&Yao24, 2=KC91, 3=Hirschmann22
    s_redox=1,                    # 1=Sun&Yao24, 2=Nash19, 3=Jugo10, 4=ONeill22, 5=Boulliung23
    scss=1,                       # 1=Blanchard21, 2=Fortin15, 3=Smythe17, 4=ONeill21
    sulfide_cap=1,                # 1=Nzotta99, 2=ONeill21, 3=Boulliung23
    co2_sol=1, h2o_sol=1, co_sol=1,

    # ── Numerical ───────────────────────────────────────────
    adiabatic=0,                  # 0 = isothermal
    solver=2,                     # 1=lsqnonlin, 2=fsolve
    gas_behavior=1,               # 1=real gas, 2=ideal
    o2_balance=0,                 # 0 = total O balanced, 1 = fixed fO2 buffer

    # ── Redox selection ─────────────────────────────────────
    redox_option="Fe3+/FeT",      # "logfO2" | "dFMQ" | "Fe3+/FeT" | "S6+/ST"
    redox_source="auto",          # "auto" | "fe3fet" | "dfmq" | "dnno" | "kc91_from_buffer"

    # ── Sat-pressure search ─────────────────────────────────
    p_start_kbar=3.0,
    p_final_kbar=0.001,
    n_steps=100,
    timeout=300,                  # MATLAB subprocess timeout (seconds)
)

## SulfurX

SulfurX uses a small set of integer/float toggles plus a nested `SulfurXSulfideConfig` describing the sulfide-phase composition (Fe / Ni / Cu / O / S wt% of the sulfide, not the melt).

In [ ]:
from volcatenate.config import SulfurXConfig, SulfurXSulfideConfig

config.sulfurx = SulfurXConfig(
    path="",                       # auto-detected; set if needed
    coh_model=0,                   # 0 = Iacono-Marziano, 1 = VolatileCalc
    slope_h2o=-0.3396,             # K2O–H2O calibration: K2O = a·H2O + b
    constant_h2o=2.7,
    n_steps=600,                   # pressure-grid steps for degassing
    fo2_tracker=1,                 # 0 = buffered fO2, 1 = redox evolution
    s_fe_choice=1,                 # 0 = Nash, 1 = O'Neill & Mavrogenes
    sigma=0.005,                   # log10fO2 tolerance for redox calc
    sulfide_pre=0,                 # sulfide precipitation toggle
    crystallization=0,             # 0 = no crystallization
    open_degassing=0,              # 0 = closed, 1 = open
    d34s_initial=0.0,              # initial bulk d34S
    sulfide=SulfurXSulfideConfig(
        fe=65.43, ni=0.0, cu=0.0, o=0.0, s=36.47,
    ),
)

## Per-sample overrides

Every backend config has an `overrides` dict keyed by sample name. Listed fields override the global value just for that sample. Useful when one inclusion in a batch needs special treatment.

In [ ]:
config.evo.overrides = {
    "MORB": {"p_start": 5000, "gas_system": "coh"},
}
config.magec.overrides = {
    "MORB": {"p_start_kbar": 8.0},
}
config.volfe.overrides = {
    "MORB": {"gassing_style": "open"},
}

## Save / load YAML

Configs round-trip through YAML. The bundled `default_config.yaml` is generated by the same writer and includes inline comments and per-section preambles describing what's hardcoded vs. comp-derived.

In [ ]:
from volcatenate.config import save_config, load_config

save_config(config, "my_full_config.yaml")
round_trip = load_config("my_full_config.yaml")
print("sulfide.fe round-trips:", round_trip.sulfurx.sulfide.fe)
print("override survives:    ", round_trip.evo.overrides["MORB"]["p_start"])

## Run + inspect resolved inputs

When you set `save_bundle`, volcatenate writes a JSON record of the run that includes per-sample, per-backend `resolved_inputs` — exactly what got handed to each model. The same data is also dropped as YAML sidecar files under `<output_dir>/resolved_inputs/<sample>/<backend>.yaml` for diff-friendly reading.

In [ ]:
import json, os
from volcatenate.core import calculate_saturation_pressure

config.output_dir = "morb_run"
config.save_bundle = os.path.join(config.output_dir, "bundle.json")

result = calculate_saturation_pressure(
    [morb], models=["EVo"], config=config,
)

bundle = json.load(open(config.save_bundle))
evo_resolved = bundle["resolved_inputs"]["MORB"]["EVo"]
print("What EVo's env.yaml actually got:")
for k in ["COMPOSITION", "GAS_SYS", "P_START", "FO2_buffer_SET", "WTH2O_START"]:
    print(f"  {k}: {evo_resolved['env'][k]}")

## Where to go next

- The [config propagation reference](../config_options.md) explains, per backend, what each field above actually does inside the model — including which values are pulled from the sample composition rather than from this config, and what fallback chains apply when a redox indicator you requested isn't available on the sample.
- The [run-bundles guide](../run_bundles.md) shows how to use the JSON bundle plus `resolved_inputs` for full reproducibility (including `replay`).
- The bundled `default_config.yaml` (run `volcatenate init-config` to copy it) carries inline comments and per-backend preambles that match this notebook field-for-field.